In [51]:
import wrds
import pandas as pd
import numpy as np
from typing import Tuple, List, Dict
from pathlib import Path

def g_cmp(d: wrds.Connection, g: str) -> pd.DataFrame:
    """Retrieves global fundamental data from Compustat Global using GVKEY.

    Output schema is normalized to use 'ni' as the net income column name
    (aliased from Compustat Global's native 'nicon') so downstream merges
    and feature engineering can be written once regardless of whether data
    came from the global or North America file.

    Note on datafmt: Compustat Global stores filings under several format
    codes. Issuers with long histories and restatements (BHP among them)
    appear under 'HIST_STD', the historically-preserved standardized
    format, rather than the plain 'STD' used for more recent entries.
    If you run this loader against a different issuer and it returns zero
    rows, diagnose with:
        SELECT DISTINCT indfmt, datafmt, popsrc, consol, COUNT(*)
        FROM comp.g_funda WHERE gvkey = %(g)s GROUP BY 1,2,3,4

    Args:
        d: Active WRDS connection.
        g: Compustat GVKEY (zero-padded 6-character string).

    Returns:
        DataFrame sorted by datadate. Columns: datadate, curcd, at, lt,
        ni (aliased from nicon), revt. Roughly one row per fiscal year.
    """
    q = """SELECT datadate, curcd, at, lt, nicon AS ni, revt
           FROM comp.g_funda
           WHERE gvkey = %(g)s
             AND indfmt = 'INDL' AND datafmt = 'HIST_STD'
             AND popsrc = 'I' AND consol = 'C'
           ORDER BY datadate ASC"""
    return d.raw_sql(q, params={'g': g}, date_cols=['datadate'])

def g_ibs_int(d: wrds.Connection, t: str) -> pd.DataFrame:
    """Retrieves international consensus EPS estimates from IBES summary file.

    Returns FY1 and FY2 annual EPS consensus from ibes.statsum_epsint (the
    split-adjusted international summary file). The fpi column distinguishes
    forecast horizons: '1'-'5' are annual (FY1 through FY5), 'A'-'D' are
    interim half-year estimates, '0' is long-term growth. We keep only the
    annual horizons because interims carry a different fiscal period
    convention that would break time-series merges downstream.

    Note: fpi is stored as a fixed-width character column, so values may
    carry trailing whitespace (e.g. '1 ' rather than '1'). TRIM() is
    required to match reliably — naive equality on '1' silently returns
    zero rows. Similarly, measure is trimmed for defensive consistency.

    Args:
        d: Active WRDS connection.
        t: International IBES ticker (e.g. '@BHP' for BHP Group Ltd).

    Returns:
        DataFrame sorted by statpers. Columns: statpers, fpedats, curcode,
        meanest, medest, numest, stdev, highest, lowest, fpi. Two rows per
        statpers (one for fpi='1', one for fpi='2').
    """
    q = """SELECT statpers, fpedats, curcode, meanest, medest, numest,
                  stdev, highest, lowest, TRIM(fpi) AS fpi
           FROM ibes.statsum_epsint
           WHERE ticker = %(t)s
             AND TRIM(measure) = 'EPS'
             AND TRIM(fpi) IN ('1', '2')
           ORDER BY statpers ASC"""
    return d.raw_sql(q, params={'t': t}, date_cols=['statpers', 'fpedats'])

def g_crs(d: wrds.Connection, p: int, s: str, e: str) -> pd.DataFrame:
    """Retrieves daily stock price, return, and volume from CRSP Version 2 using PERMNO."""
    q = """SELECT dlycaldt,
                  ABS(dlyprc) AS dlyprc,
                  dlyret, dlyvol
           FROM crsp.dsf_v2
           WHERE permno = %(p)s
             AND dlycaldt >= %(s)s
             AND dlycaldt <= %(e)s
           ORDER BY dlycaldt ASC"""
    return d.raw_sql(q, params={'p': p, 's': s, 'e': e}, date_cols=['dlycaldt'])

def g_evt(d: wrds.Connection, c: str) -> pd.DataFrame:
    """Retrieves self-describing key developments from Capital IQ.

    Args:
        d: Active WRDS connection.
        c: Capital IQ CompanyID.

    Returns:
        DataFrame sorted by announcedate containing event details and category taxonomy.
    """
    q = """
        SELECT
            a.announcedate,
            a.keydeveventtypeid,
            t.keydeveventtypename AS eventtype,
            t.keydevcategoryname AS category,
            b.headline
        FROM ciq.wrds_keydev a
        JOIN ciq.ciqkeydev b ON a.keydevid = b.keydevid
        LEFT JOIN ciq.ciqkeydevcategorytype t ON a.keydeveventtypeid = t.keydeveventtypeid
        WHERE a.companyid = %(c)s
        ORDER BY a.announcedate ASC
    """
    return d.raw_sql(q, params={'c': c}, date_cols=['announcedate'])

def g_ff(d: wrds.Connection, s: str, e: str) -> pd.DataFrame:
    """Retrieves Fama-French 5-Factor daily data."""
    q = f"SELECT date, mktrf, smb, hml, rmw, cma, rf FROM ff.fivefactors_daily WHERE date >= '{s}' AND date <= '{e}' ORDER BY date ASC"
    try:
        return d.raw_sql(q, date_cols=['date']).rename(columns={'date': 'ff_date'})
    except Exception:
        return pd.DataFrame(columns=['ff_date', 'mktrf', 'smb', 'hml', 'rmw', 'cma', 'rf'])

def g_esg(d: wrds.Connection, t: str) -> pd.DataFrame:
    """Retrieves ESG/Governance proxies dynamically."""
    q = f"SELECT meeting_date as as_of_date, female_directors, minority_directors FROM iss_directors_global.company_diversity WHERE ticker = '{t}' ORDER BY meeting_date ASC"
    try:
        return d.raw_sql(q, date_cols=['as_of_date'])
    except Exception:
        return pd.DataFrame(columns=['as_of_date', 'female_directors', 'minority_directors'])

def g_bdx(d: wrds.Connection, t: str) -> pd.DataFrame:
    """Retrieves BoardEx Director Network data."""
    q = f"SELECT a.annual_report_date, AVG(b.network_size) as avg_board_network FROM boardex_row.row_wrds_org_summary a JOIN boardex_row.row_wrds_individual_networks b ON a.companyid = b.companyid WHERE a.ticker = '{t}' GROUP BY a.annual_report_date ORDER BY a.annual_report_date ASC"
    try:
        return d.raw_sql(q, date_cols=['annual_report_date'])
    except Exception:
        return pd.DataFrame(columns=['annual_report_date', 'avg_board_network'])

def g_shv(d: wrds.Connection, t: str, s: str, e: str) -> pd.DataFrame:
    """Retrieves short volume and interest data."""
    q = f"SELECT date, shortint FROM comp.sec_shortint WHERE tic = '{t}' AND date >= '{s}' AND date <= '{e}' ORDER BY date ASC"
    try:
        return d.raw_sql(q, date_cols=['date']).rename(columns={'date': 'shv_date'})
    except Exception:
        return pd.DataFrame(columns=['shv_date', 'shortint'])

def g_bta(d: wrds.Connection, p: int, s: str, e: str) -> pd.DataFrame:
    """Retrieves WRDS Beta Suite daily data."""
    q = f"SELECT date, beta FROM betasuite.beta_daily WHERE permno = {p} AND date >= '{s}' AND date <= '{e}' ORDER BY date ASC"
    try:
        return d.raw_sql(q, date_cols=['date']).rename(columns={'date': 'bta_date'})
    except Exception:
        return pd.DataFrame(columns=['bta_date', 'beta'])

def g_fac(d: wrds.Connection, s: str, e: str) -> pd.DataFrame:
    """Retrieves WRDS daily factors."""
    q = f"SELECT date, mkt, smb, hml, umd FROM wrdsapps.factors_daily WHERE date >= '{s}' AND date <= '{e}' ORDER BY date ASC"
    try:
        return d.raw_sql(q, date_cols=['date']).rename(columns={'date': 'fac_date'})
    except Exception:
        return pd.DataFrame(columns=['fac_date', 'mkt', 'smb', 'hml', 'umd'])

def b_pipe(c: pd.DataFrame, i: pd.DataFrame, r: pd.DataFrame, e: pd.DataFrame, ff: pd.DataFrame, esg: pd.DataFrame, bdx: pd.DataFrame, shv: pd.DataFrame, bta: pd.DataFrame, fac: pd.DataFrame) -> pd.DataFrame:
    """Builds an aligned, forward-filled feature matrix mapping off-hours events to trading sessions.

    Args:
        c: Compustat fundamentals DataFrame.
        i: IBES estimates DataFrame.
        r: CRSP daily returns DataFrame.
        e: Capital IQ events DataFrame.
        ff: Fama-French factors DataFrame.
        esg: ISS ESG DataFrame.
        bdx: BoardEx DataFrame.
        shv: Short Volume DataFrame.
        bta: Beta Suite DataFrame.
        fac: Factors DataFrame.

    Returns:
        pd.DataFrame: Integrated and aligned feature matrix.
    """
    def _fmt(df: pd.DataFrame, col: str) -> pd.DataFrame:
        if not df.empty:
            df[col] = pd.to_datetime(df[col], errors='coerce').dt.normalize()
            return df.dropna(subset=[col]).sort_values(col)
        return df

    r = _fmt(r, 'dlycaldt')
    c = _fmt(c, 'datadate')
    i = _fmt(i, 'statpers')
    e = _fmt(e, 'announcedate')
    ff = _fmt(ff, 'ff_date')
    esg = _fmt(esg, 'as_of_date')
    bdx = _fmt(bdx, 'annual_report_date')
    shv = _fmt(shv, 'shv_date')
    bta = _fmt(bta, 'bta_date')
    fac = _fmt(fac, 'fac_date')

    if not i.empty:
        i['fpi'] = pd.to_numeric(i['fpi'], errors='coerce').astype('Int8')
        i = i.dropna(subset=['fpi'])
        i_pvt = i.groupby(['statpers', 'fpi'])[['meanest', 'medest', 'numest', 'stdev', 'highest', 'lowest', 'fpedats', 'curcode']].last().unstack()
        i_pvt.columns = [f"{col[0]}_fy{col[1]}" for col in i_pvt.columns]
        i = i_pvt.reset_index()

    m = pd.merge_asof(r, c, left_on='dlycaldt', right_on='datadate', direction='backward')

    if not i.empty: m = pd.merge_asof(m, i, left_on='dlycaldt', right_on='statpers', direction='backward')
    if not ff.empty: m = pd.merge_asof(m, ff, left_on='dlycaldt', right_on='ff_date', direction='backward')
    if not esg.empty: m = pd.merge_asof(m, esg, left_on='dlycaldt', right_on='as_of_date', direction='backward')
    if not bdx.empty: m = pd.merge_asof(m, bdx, left_on='dlycaldt', right_on='annual_report_date', direction='backward')
    if not shv.empty: m = pd.merge_asof(m, shv, left_on='dlycaldt', right_on='shv_date', direction='backward')
    if not bta.empty: m = pd.merge_asof(m, bta, left_on='dlycaldt', right_on='bta_date', direction='backward')
    if not fac.empty: m = pd.merge_asof(m, fac, left_on='dlycaldt', right_on='fac_date', direction='backward')

    if not e.empty:
        e_g = e.groupby('announcedate').agg({
            'keydeveventtypeid': list,
            'eventtype': list,
            'category': list,
            'headline': list
        }).reset_index().sort_values('announcedate')

        e_m = pd.merge_asof(e_g, r[['dlycaldt']], left_on='announcedate', right_on='dlycaldt', direction='forward')

        e_f = e_m.dropna(subset=['dlycaldt']).groupby('dlycaldt').agg({
            'keydeveventtypeid': 'sum',
            'eventtype': 'sum',
            'category': 'sum',
            'headline': 'sum'
        }).reset_index()

        m = pd.merge(m, e_f, on='dlycaldt', how='left')

    d_cols = ['datadate', 'statpers', 'ff_date', 'as_of_date', 'annual_report_date', 'shv_date', 'bta_date', 'fac_date']
    m.drop(columns=[col for col in d_cols if col in m.columns], inplace=True)

    return m

In [18]:
def g_sch(d: wrds.Connection, k: str) -> pd.DataFrame:
    """Lists schema.table pairs whose schema name matches a keyword."""
    q = "SELECT table_schema, table_name FROM information_schema.tables WHERE table_schema LIKE %(k)s ORDER BY table_schema, table_name"
    return d.raw_sql(q, params={'k': f'%{k}%'})

def g_col(d: wrds.Connection, schema: str, table: str) -> pd.DataFrame:
    """Lists columns and their data types for a given schema.table."""
    q = "SELECT column_name, data_type FROM information_schema.columns WHERE table_schema = %(s)s AND table_name = %(t)s ORDER BY ordinal_position"
    return d.raw_sql(q, params={'s': schema, 't': table})

In [19]:
def g_sch(d: wrds.Connection, k: str) -> pd.DataFrame:
    """Lists schema.table pairs whose schema name matches a keyword."""
    q = "SELECT table_schema, table_name FROM information_schema.tables WHERE table_schema LIKE %(k)s ORDER BY table_schema, table_name"
    return d.raw_sql(q, params={'k': f'%{k}%'})

def g_col(d: wrds.Connection, schema: str, table: str) -> pd.DataFrame:
    """Lists columns and their data types for a given schema.table."""
    q = "SELECT column_name, data_type FROM information_schema.columns WHERE table_schema = %(s)s AND table_name = %(t)s ORDER BY ordinal_position"
    return d.raw_sql(q, params={'s': schema, 't': table})

In [59]:
usr      = "zackienzle1"
gvkey    = "100712"
permno   = 23169
ibes_tic = "@WPL"
ciq_id   = "873963"
us_tic   = "WDS"
st       = "1900-01-01"
ed       = "2026-04-10"

db = wrds.Connection(wrds_username=usr)

Loading library list...
Done


In [53]:
print(f"Calling g_crs with permno={permno}, start={st}, end={ed}")
df_crs = g_crs(db, permno, st, ed)
print(f"Rows returned: {len(df_crs)}")
if not df_crs.empty:
    print(f"Date range returned: {df_crs['dlycaldt'].min().date()} to {df_crs['dlycaldt'].max().date()}")
    print(df_crs.head(5).to_string(index=False))
    print(df_crs.tail(5).to_string(index=False))
else:
    print("No rows returned from g_crs for these arguments.")

db.close()

Calling g_crs with permno=23169, start=1900-01-01, end=2026-04-10
Rows returned: 899
Date range returned: 2022-06-02 to 2025-12-31
  dlycaldt  dlyprc    dlyret     dlyvol
2022-06-02   23.15      <NA>  1422570.0
2022-06-03    23.0 -0.006479  1233276.0
2022-06-06   23.76  0.033044  1651320.0
2022-06-07   24.11  0.014731  2265246.0
2022-06-08   25.02  0.037744  1899037.0
  dlycaldt  dlyprc    dlyret    dlyvol
2025-12-24   15.42 -0.003876  424350.0
2025-12-26   15.36 -0.003891  776072.0
2025-12-29   15.52  0.010417  694558.0
2025-12-30   15.62  0.006443  543998.0
2025-12-31   15.59 -0.001921  630336.0


In [54]:
df_crs

,dlycaldt,dlyprc,dlyret,dlyvol
0,2022-06-02,23.15,<NA>,1422570.0
1,2022-06-03,23.0,-0.006479,1233276.0
2,2022-06-06,23.76,0.033044,1651320.0
3,2022-06-07,24.11,0.014731,2265246.0
4,2022-06-08,25.02,0.037744,1899037.0
...,...,...,...,...
894,2025-12-24,15.42,-0.003876,424350.0
895,2025-12-26,15.36,-0.003891,776072.0
896,2025-12-29,15.52,0.010417,694558.0
897,2025-12-30,15.62,0.006443,543998.0


In [55]:
df_crs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 899 entries, 0 to 898
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   dlycaldt  899 non-null    datetime64[ns]
 1   dlyprc    899 non-null    Float64       
 2   dlyret    898 non-null    Float64       
 3   dlyvol    899 non-null    Float64       
dtypes: Float64(3), datetime64[ns](1)
memory usage: 30.9 KB


In [24]:
def s_csv(d_m: Dict[str, pd.DataFrame], d_p: str = "../data") -> None:
    """Saves multiple DataFrames to CSV efficiently."""
    p = Path(d_p)
    p.mkdir(parents=True, exist_ok=True)
    for k, v in d_m.items():
        if not v.empty:
            v.to_csv(p / f"{k}.csv", index=False, chunksize=100000)

In [56]:
d_out = {
    "crsp_daily_prices": df_crs,
}

s_csv(d_out)
print(f"Standalone CRSP extract saved. Shape: {df_crs.shape}")

Standalone CRSP extract saved. Shape: (899, 4)


In [26]:
# import wrds
# import pandas as pd

# def ext_evt(db: wrds.Connection, out_p: str = "ciq_all_event_types.csv") -> pd.DataFrame:
#     q = """
#         SELECT keydeveventtypeid, eventtypename AS event_name
#         FROM ciq.ciqeventtype
#         ORDER BY keydeveventtypeid
#     """
#     df = db.raw_sql(q)
#     df.to_csv(out_p, index=False)
#     return df

# def d_cov(db: wrds.Connection, s: str, t: str, d_c: str, i_c: str, i_v: str) -> pd.DataFrame:
#     q = f"""
#         SELECT MIN({d_c}) AS min_date, MAX({d_c}) AS max_date, COUNT(1) AS n_obs
#         FROM {s}.{t}
#         WHERE {i_c} = '{i_v}'
#     """
#     return db.raw_sql(q)

# def d_lnk(db: wrds.Connection, gvkey: str) -> pd.DataFrame:
#     q = f"""
#         SELECT lpermno, linkdt, linkenddt, linktype, linkprim
#         FROM crsp.ccmxpf_lnkhist
#         WHERE gvkey = '{gvkey}'
#         ORDER BY linkdt ASC
#     """
#     return db.raw_sql(q)

In [27]:
# df_evt = ext_evt(db)
# print(df_evt.head().to_string(index=False))
# print("\n")

# df_cov = d_cov(db, "ciq", "wrds_keydev", "announcedate", "companyid", ciq_id)
# print(df_cov.to_string(index=False))
# print("\n")

# df_lnk = d_lnk(db, gvkey)
# print(df_lnk.to_string(index=False))

In [28]:
def fetch_event_taxonomy(db: wrds.Connection) -> pd.DataFrame:
    """Fetches the full CIQ key-development event-type dictionary with category hierarchy.

    Returns the fine-grained 139-row taxonomy joining event type IDs to both
    the leaf name (``keydeveventtypename``) and the parent category
    (``keydevcategoryname``), which serves as the semantic pooling layer.
    """
    q = """SELECT keydeveventtypeid,
                  keydeveventtypename AS event_name,
                  keydevcategoryid,
                  keydevcategoryname  AS category_name
           FROM ciq.ciqkeydevcategorytype
           ORDER BY keydeveventtypeid"""
    df = db.raw_sql(q, params={})
    df["keydeveventtypeid"] = df["keydeveventtypeid"].astype(int)
    return df


db = wrds.Connection(wrds_username=usr)
taxonomy = fetch_event_taxonomy(db)
db.close()

taxonomy.to_csv("../data/ciq_event_taxonomy.csv", index=False)
print(f"Taxonomy: {len(taxonomy)} event types across {taxonomy['category_name'].nunique()} parent categories")
print(taxonomy.to_string(index=False))

Loading library list...
Done
Taxonomy: 139 event types across 13 parent categories
 keydeveventtypeid                                           event_name  keydevcategoryid                                  category_name
                 1                               Seeking to Sell/Divest               3.0                         Potential Transactions
                 3                     Seeking Acquisitions/Investments               3.0                         Potential Transactions
                 5                           Seeking Financing/Partners               3.0                         Potential Transactions
                 7                                   Bankruptcy - Other               6.0 Results Announcements/Corporate Communications
                 7                                   Bankruptcy - Other              12.0                             Bankruptcy Updates
                11                                  Delayed SEC Filings               5.0      

In [57]:
def g_permno(d: wrds.Connection, gvkey: str, asof: str | None = None) -> pd.DataFrame:
    g = str(gvkey).zfill(6)
    if asof is None:
        q = (
            "SELECT lpermno AS permno, linkdt, linkenddt, linktype, linkprim "
            "FROM crsp.ccmxpf_lnkhist WHERE gvkey = %(g)s ORDER BY linkdt"
        )
        return d.raw_sql(q, params={"g": g}, date_cols=["linkdt", "linkenddt"])
    q = (
        "SELECT lpermno AS permno, linkdt, linkenddt, linktype, linkprim "
        "FROM crsp.ccmxpf_lnkhist WHERE gvkey = %(g)s "
        "AND linkdt <= %(a)s::date AND linkenddt >= %(a)s::date "
        "ORDER BY (linkprim = 'P') DESC, linkdt DESC"
    )
    return d.raw_sql(q, params={"g": g, "a": asof}, date_cols=["linkdt", "linkenddt"])


def g_dsf_rng(d: wrds.Connection, permnos: list[int]) -> pd.DataFrame:
    permnos = [int(p) for p in permnos]
    if not permnos:
        return pd.DataFrame(columns=["permno", "min_dt", "max_dt", "n"])
    ins = ",".join(str(p) for p in permnos)
    q = (
        f"SELECT permno, MIN(dlycaldt) AS min_dt, MAX(dlycaldt) AS max_dt, COUNT(*)::bigint AS n "
        f"FROM crsp.dsf_v2 WHERE permno IN ({ins}) GROUP BY permno"
    )
    return d.raw_sql(q, date_cols=["min_dt", "max_dt"])

In [62]:
h = g_permno(db, "100712")
p = g_permno(db, "100712", "2026-04-10")
cov = g_dsf_rng(db, h["permno"].dropna().unique().astype("int64").tolist())

In [63]:
cov

,permno,min_dt,max_dt,n
0,23169,2022-06-02,2025-12-31,899


In [67]:
def g_sn(d: wrds.Connection, tic: str) -> pd.DataFrame:
    q = (
        "SELECT permno, namedt, nameenddt, ticker, comnam, exchcd "
        "FROM crsp.stocknames WHERE TRIM(ticker) = %(t)s "
        "ORDER BY permno, namedt"
    )
    return d.raw_sql(q, params={"t": tic.upper()}, date_cols=["namedt", "nameenddt"])

In [68]:
sn_wds = g_sn(db, "WDS")
sn_wpl = g_sn(db, "WPL")

In [69]:
asof = pd.Timestamp("2026-04-10")
def active_permno(sn: pd.DataFrame, asof: pd.Timestamp) -> pd.DataFrame:
    x = sn.copy()
    x["namedt"] = pd.to_datetime(x["namedt"])
    x["nameenddt"] = pd.to_datetime(x["nameenddt"])
    asof = pd.Timestamp(asof).normalize()
    y = x.loc[x["namedt"] <= asof].sort_values(["permno", "namedt"])
    return y.groupby("permno", as_index=False).tail(1)

active_permno(sn_wds, asof)



,permno,namedt,nameenddt,ticker,comnam,exchcd
0,23169,2022-06-02,2024-12-31,WDS,WOODSIDE ENERGY GROUP LTD,1
3,41400,1968-06-17,1978-12-12,WDS,WOODS CORP,1


In [70]:
def g_crsp_id(
    d: wrds.Connection,
    gvkey: str,
    asof: str | None = None,
) -> pd.DataFrame:
    g = str(gvkey).zfill(6)
    q_lnk = (
        "SELECT lpermno AS permno, linkdt, linkenddt, linktype, linkprim "
        "FROM crsp.ccmxpf_lnkhist WHERE gvkey = %(g)s ORDER BY linkdt"
    )
    lnk = d.raw_sql(q_lnk, params={"g": g}, date_cols=["linkdt", "linkenddt"])
    if lnk.empty:
        return lnk
    permnos = sorted({int(p) for p in lnk["permno"].dropna().tolist()})
    ins = ",".join(str(p) for p in permnos)
    q_cov = (
        f"SELECT permno, MIN(dlycaldt) AS min_dt, MAX(dlycaldt) AS max_dt, "
        f"COUNT(*)::bigint AS n FROM crsp.dsf_v2 WHERE permno IN ({ins}) GROUP BY permno"
    )
    cov = d.raw_sql(q_cov, date_cols=["min_dt", "max_dt"])
    if asof is None:
        q_sn = (
            f"SELECT DISTINCT ON (permno) permno, namedt, nameenddt, "
            f"TRIM(ticker) AS ticker, comnam, exchcd FROM crsp.stocknames "
            f"WHERE permno IN ({ins}) ORDER BY permno, namedt DESC"
        )
        sn = d.raw_sql(q_sn, date_cols=["namedt", "nameenddt"])
    else:
        q_sn = (
            f"SELECT DISTINCT ON (permno) permno, namedt, nameenddt, "
            f"TRIM(ticker) AS ticker, comnam, exchcd FROM crsp.stocknames "
            f"WHERE permno IN ({ins}) AND namedt <= %(a)s::date "
            f"ORDER BY permno, namedt DESC"
        )
        sn = d.raw_sql(q_sn, params={"a": asof}, date_cols=["namedt", "nameenddt"])
    out = lnk.merge(cov, on="permno", how="left").merge(sn, on="permno", how="left")
    out["_p"] = out["linkprim"].eq("P")
    return out.sort_values(
        ["max_dt", "_p", "linkdt"],
        ascending=[False, False, True],
        na_position="last",
    ).drop(columns="_p")

In [71]:
df = g_crsp_id(db, "100712", "2026-04-10")
permno = int(df.iloc[0]["permno"])

In [72]:
permno

23169